# Support Ticket Resolution Retrieval

This notebook implements a historical resolution retrieval system. It filters the dataset to resolved tickets, indexes them using TF-IDF and Word2Vec embedding models, and compares retrieval quality using Cosine Similarity.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import os

try:
    from gensim.models import Word2Vec
    HAS_GENSIM = True
except ImportError:
    HAS_GENSIM = False
    print("Warning: gensim not installed. Word2Vec experiment will be skipped.")

### Load and Filter Dataset
Filter the tickets to only include those that have a valid resolution.

In [ ]:
df = pd.read_csv("../data/processed/clean_tickets.csv")
print("Original dataset shape:", df.shape)

# Filter resolved tickets
resolved_df = df[df["Resolution"].notna() & (df["Resolution"].str.strip() != "")].copy()
resolved_df["processed_text"] = resolved_df["processed_text"].fillna("")
print("Resolved tickets shape:", resolved_df.shape)

### TF-IDF + Cosine Similarity Baseline

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(resolved_df["processed_text"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
def retrieve_similar_tfidf(query_text, top_k=3):
    clean_query = query_text.lower()
    clean_query = re.sub(r"[^a-zA-Z\s]", "", clean_query)
    clean_query = " ".join(clean_query.split())
    
    query_vector = tfidf_vectorizer.transform([clean_query])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "subject": resolved_df.iloc[idx]["Ticket Subject"],
            "description": resolved_df.iloc[idx]["Ticket Description"],
            "resolution": resolved_df.iloc[idx]["Resolution"],
            "similarity": similarities[idx]
        })
    return results

### Word2Vec + Cosine Similarity Experiment

In [ ]:
if HAS_GENSIM:
    sentences = [text.split() for text in resolved_df["processed_text"]]
    w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4, seed=42)
    print("Word2Vec vocabulary size:", len(w2v_model.wv))
else:
    print("Skipping Word2Vec model training (gensim not installed).")

In [ ]:
if HAS_GENSIM:
    def get_average_w2v(tokens, model, vector_size=100):
        vectors = [model.wv[word] for word in tokens if word in model.wv]
        if len(vectors) == 0:
            return np.zeros(vector_size)
        return np.mean(vectors, axis=0)

    document_vectors = np.array([get_average_w2v(tokens, w2v_model) for tokens in sentences])
    print("Document vectors shape:", document_vectors.shape)
else:
    print("Skipping document vector creation (gensim not installed).")

In [ ]:
def retrieve_similar_w2v(query_text, top_k=3):
    if not HAS_GENSIM:
        return [{
            "subject": "N/A",
            "description": "N/A",
            "resolution": "Gensim/Word2Vec not available in this environment.",
            "similarity": 0.0
        }] * top_k
    clean_query = query_text.lower()
    clean_query = re.sub(r"[^a-zA-Z\s]", "", clean_query)
    clean_query = " ".join(clean_query.split())
    query_tokens = clean_query.split()
    
    query_vector = get_average_w2v(query_tokens, w2v_model).reshape(1, -1)
    similarities = cosine_similarity(query_vector, document_vectors).flatten()
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "subject": resolved_df.iloc[idx]["Ticket Subject"],
            "description": resolved_df.iloc[idx]["Ticket Description"],
            "resolution": resolved_df.iloc[idx]["Resolution"],
            "similarity": similarities[idx]
        })
    return results

### Comparison on Typical Queries

In [ ]:
sample_queries = [
    "I cannot log in to my account, it says incorrect password",
    "How do I request a refund for my last purchase?",
    "The software crashes every time I open the dashboard"
]

for q in sample_queries:
    print(f"\nQuery: '{q}'")
    print("--- TF-IDF Results ---")
    tfidf_res = retrieve_similar_tfidf(q)
    for r in tfidf_res:
        print(f"Subj: {r['subject']} | Sim: {r['similarity']:.4f} | Res: {r['resolution'][:100]}...")
    
    print("--- Word2Vec Results ---")
    w2v_res = retrieve_similar_w2v(q)
    for r in w2v_res:
        print(f"Subj: {r['subject']} | Sim: {r['similarity']:.4f} | Res: {r['resolution'][:100]}...")

### Save Final Retrieval Model
We select TF-IDF as the final retrieval approach as it provides precise keyword matching (important for matching specific error messages/product terms in support tickets) and is robust for this scale of text data.

In [ ]:
selected_method = "tfidf"

os.makedirs("../models", exist_ok=True)
joblib.dump(selected_method, "../models/retrieval_method.pkl")
joblib.dump(tfidf_vectorizer, "../models/retrieval_tfidf.pkl")
joblib.dump(tfidf_matrix, "../models/retrieval_matrix.pkl")
joblib.dump(resolved_df[["Ticket Subject", "Ticket Description", "Resolution"]], "../models/retrieval_tickets.pkl")
print("Saved retrieval artifacts to models/ directory.")